# Baseline Dataset EDA (CardSim)

Exploratory data analysis of `data/raw/cardsim_base.parquet` to establish
baseline distributions and calibrate downstream Card Testing and Bust-Out
campaign injectors.

Preprocessing audit checklist:
1. Type verification
2. Discretization and normalization
3. Quantitative inconsistencies
4. Sentinels, outliers, and robust estimators
5. Missing values

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Colorblind-safe palette with print-friendly gridlines
sns.set_theme(style="whitegrid", palette="muted")


def _find_project_root() -> Path:
    """Walk upward from cwd until pyproject.toml is found."""
    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError(f"Could not locate pyproject.toml above {cwd}")


PROJECT_ROOT = _find_project_root()
DATASET_PATH = PROJECT_ROOT / "data" / "raw" / "cardsim_base.parquet"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(filename: str) -> None:
    """Save the active Matplotlib figure to reports/figures and display it."""
    plt.savefig(FIGURES_DIR / filename, bbox_inches="tight", dpi=300)
    plt.show()

## 1. Data Loading & Type Verification

Verify that each variable has the expected statistical type
(continuous, categorical, boolean, datetime, or identifier).

In [ ]:
df = pd.read_parquet(DATASET_PATH)

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print("\nFirst 5 rows:")
print(df.head().to_string())

print(
    f"\nTime span: {df['transaction_timestamp'].min()} -> "
    f"{df['transaction_timestamp'].max()}"
)
print(
    f"Unique cards: {df['card_id'].nunique():,} | "
    f"Unique merchants: {df['merchant_id'].nunique():,}"
)
print(f"\nDtypes (raw):\n{df.dtypes.to_string()}")

# Cast numeric identifiers to str to prevent accidental arithmetic aggregations
df["card_id"] = df["card_id"].astype(str)
df["merchant_id"] = df["merchant_id"].astype(str)

print(f"\nDtypes (cast):\n{df.dtypes.to_string()}")

print("\nCategory counts:")
for col in ("card_type", "channel", "status"):
    print(f"  {col}: {df[col].value_counts().to_dict()}")

## 2. Transaction Amount Distribution

**Goal:** Characterize normal transaction amounts to calibrate low-value
Card Testing probes and high-value Bust-Out spikes.

**Methods:** Sentinel checks, Tukey fences, robust location/scale estimators
(median, IQR, MAD, trimmed/Winsorized means), and classical vs. robust z-scores.